In [ ]:
### HERE COMES THE ENITRE ANALYSIS AND THE PLOTS I WILL CREATE
### IT WILL ACCESS THE SAME FUNCTIONS (PARTIALLY) AS THE 4_homologs_analysis_visualization.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for custom src imports later
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Default plot style
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

In [ ]:
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

df = pd.read_parquet(f"{DATA_DIR}/variants_annotated_final.parquet")
print(f"Loaded {len(df):,} variant-region assignments")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

# Also load the regions JSON — useful for context (WT sequences, etc.)
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MANUAL FIX: coordinate convention inconsistency in regions JSON
# TODO: fix at source in 16_ev_signature_predictor/src/write_bed_file.py
# or wherever genomic_coords_merged_win5.json gets generated.
#
# Issue: most regions use inclusive-inclusive (len = end - start + 1),
# but 12 regions use half-open (len = end - start). 6 regions are broken
# (seq_len 0 or 1). This patches them for analysis only.
# ═══════════════════════════════════════════════════════════════════════════

# Inspect the half-open regions
halfopen_region_ids = []
broken_region_ids = []
for r in regions:
    start, end = r["prot_region"]
    seq_len = len(r["prot_seq"])
    if seq_len < 2:
        broken_region_ids.append(r["region_id"])
    elif seq_len == end - start:
        halfopen_region_ids.append(r["region_id"])

print(f"Half-open regions to patch: {len(halfopen_region_ids)}")
for rid in halfopen_region_ids:
    r = region_by_id[rid]
    print(f"  {rid}: prot_region={r['prot_region']}, "
          f"seq='{r['prot_seq']}' (len {len(r['prot_seq'])})")

print(f"\nBroken regions to drop: {len(broken_region_ids)}")
for rid in broken_region_ids:
    r = region_by_id[rid]
    print(f"  {rid}: prot_region={r['prot_region']}, seq_len={len(r['prot_seq'])}")


# ═══════════════════════════════════════════════════════════════════════════
# MANUAL FIX: apply the patch
# ═══════════════════════════════════════════════════════════════════════════

# 1. Patch half-open regions — shift to 1-based inclusive
for rid in halfopen_region_ids:
    r = region_by_id[rid]
    old_start, old_end = r["prot_region"]
    # Half-open [0, N) -> inclusive-inclusive 1-based [1, N]
    r["prot_region"] = [old_start + 1, old_end]
    # Sanity check
    assert len(r["prot_seq"]) == r["prot_region"][1] - r["prot_region"][0] + 1, \
        f"Patch failed for {rid}"

# 2. Drop broken regions
for rid in broken_region_ids:
    del region_by_id[rid]

# 3. Flag affected variants for traceability
df["coord_patched"] = df["region_id"].isin(halfopen_region_ids)
df["coord_dropped"] = df["region_id"].isin(broken_region_ids)
print(f"Variants in patched regions: {df['coord_patched'].sum()}")
print(f"Variants in dropped regions: {df['coord_dropped'].sum()}")

# 4. Remove variants in dropped regions from df
df = df[~df["coord_dropped"]].copy()
print(f"Remaining variants after drop: {len(df):,}")

# 5. Update region_start_aa in df for patched regions
patched_mask = df["region_id"].isin(halfopen_region_ids)
df.loc[patched_mask, "region_start_aa"] = df.loc[patched_mask, "region_start_aa"] + 1

# Also update region_end_aa stays the same (only start changed)
# (end was already consistent with the actual sequence end)

In [ ]:
def _check(row):
    region = region_by_id.get(row["region_id"])
    if region is None or pd.isna(row["protein_position_int"]):
        return None
    pos = int(row["protein_position_int"]) - int(row["region_start_aa"])
    if pos < 0 or pos >= len(region["prot_seq"]):
        return None
    if row["before_aa"] is None:
        return None
    # Skip insertions (before_aa is '-') and multi-residue cases
    if row["before_aa"] == "-" or len(row["before_aa"]) != 1:
        return None
    return region["prot_seq"][pos] == row["before_aa"]

df["wt_match"] = df.apply(_check, axis=1)
print(df["wt_match"].value_counts(dropna=False))

In [ ]:
# Flag reliability for RG-level analysis
df["rg_analysis_reliable"] = df["wt_match"] == True

# Any analysis requiring exact region sequence should subset on this
df_for_rg = df[df["rg_analysis_reliable"]].copy()

print(f"Total variants: {len(df):,}")
print(f"Usable for RG analysis: {len(df_for_rg):,}")
print(f"\nBreakdown by group:")
print(df_for_rg["group"].value_counts())

In [ ]:
###### NOW ANALYSIS

In [ ]:
import src.analysis_visualization.region_analysis as ra

fig, stats = ra.plot_variant_density(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, results = ra.plot_consequence_distributions(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, r = ra.plot_median_alphamissense(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
import src.analysis_visualization.rg_analysis as rga
df_rg = rga.compute_rg_disruption_columns(df_for_rg, region_by_id)
print("RG hit columns added. Summary:")
print(df_rg["hits_rg"].value_counts())

# Run each plot independently
fig_a, r_a = rga.plot_region_length(region_by_id, dataset="gnomad")
plt.show()

fig_b, r_b = rga.plot_n_rg_motifs(region_by_id, dataset="gnomad")
plt.show()

fig_c, r_c = rga.plot_rg_density(region_by_id, dataset="gnomad")
plt.show()

# D1 — density version (replaces the saturating D)
fig_d1, r_d1 = rga.plot_variants_per_rg_by_type(df_rg, region_by_id, dataset="gnomad")
plt.show()

# # D2 — AlphaMissense on RG-hitting missense
# fig_d2, r_d2 = rga.plot_median_alphamissense_on_rgs(df_rg, dataset="gnomad")
# plt.show()


In [ ]:
df_rg

In [ ]:
fig, r = rga.plot_rg_role_asymmetry(df_rg, dataset="gnomad")
plt.show()

In [ ]:
results = rga.plot_rg_change_events_stacked(df_rg, region_by_id, dataset="gnomad")

# the single event version has been commmented out in the rg-analysis, because it showed no signifincance in any plot, so here is just the stacked version

In [ ]:
# Make sure df_events is computed (already have compute_rg_change_events from before)
df_events = rga.compute_rg_change_events(df_rg, region_by_id)

# Analysis 1
loss_transition_results = rga.plot_rg_loss_transitions(df_events, dataset="gnomad")
plt.show()

# Analysis 2
cluster_results = rga.plot_isolated_vs_clustered_loss(
    df_events, region_by_id,
    window_sizes=[2,4,6],
    dataset="gnomad",
)
plt.show()

In [ ]:
gain_transition_results = rga.plot_rg_gain_transitions(df_events, dataset="gnomad")
plt.show()

In [ ]:
# # Build the null once — reused for both plots
# null_results = rga.build_enumeration_null(region_by_id, df_for_rg)

# # RG events observed vs expected
# rg_comparison = rga.plot_rg_events_observed_vs_expected(
#     df_events, null_results, dataset="gnomad",
# )
# plt.show()

# # Consequences observed vs expected
# cons_comparison = rga.plot_consequences_observed_vs_expected(
#     df_for_rg, null_results, dataset="gnomad",
# )
# plt.show()

In [ ]:
# Build null once
null_results = rga.build_enumeration_null(region_by_id, df_for_rg)

# RG events: 4 subplots (no_change / loss / gain / movement)
rg_results = rga.plot_rg_events_vs_expected_boxes(
    df_events, null_results, dataset="gnomad",
)
plt.show()

# Consequences: 3 subplots (synonymous / missense / nonsense)
cons_results = rga.plot_consequences_vs_expected_boxes(
    df_for_rg, null_results, dataset="gnomad",
)
plt.show()

In [ ]:
# Plot A: per-variant distribution, R/G-affecting only
fig_a, r_a = rga.plot_delta_rg_ratio_per_variant(df_rg, region_by_id, dataset="gnomad")
plt.show()

# Plot B: per-region mean across all missense
fig_b, r_b = rga.plot_delta_rg_ratio_per_region(df_rg, region_by_id, dataset="gnomad")
plt.show()

In [ ]:
############### PHYSCHEM

In [ ]:
import src.analysis_visualization.physchem_analysis as pca

# # Step 1: compute per-variant deltas (slow — uses multiprocessing)
# deltas_df = pca.compute_physchem_deltas(df_rg, region_by_id)

# # Save the result — expensive to recompute
# deltas_df.to_parquet(
#     "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
# )

deltas_df = pd.read_parquet("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
)

# Step 2: aggregate to per-region means for classifier features
per_region = pca.aggregate_per_region(deltas_df)

# Step 3: plot each feature individually
all_results = pca.plot_all_delta_features(per_region, dataset="gnomad")

# Bonus: per-region WT baseline features (no variants involved — useful for classifier too)
wt_features = pca.compute_wt_physchem(region_by_id)

In [ ]:
################# AMINO ACID SUBSTITUTIONS

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# All-missense matrix (single call)
enrichment = smx.run_substitution_analysis(
    df_for_rg, dataset="gnomad", min_total=5,
)

# Access individual outputs
log2_or_matrix = enrichment["log2_or"]
fdr_matrix = enrichment["fdr"]
counts_pos = enrichment["counts_pos"]

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# Side-by-side rare (AF < 1e-4) vs common (AF >= 1e-3) matrices
results = smx.plot_af_comparison_matrices(
    df_for_rg,
    af_rare_max=1e-5,
    af_common_min=1e-5,
    dataset="gnomad",
)

# Access individual results
rare_enrichment = results["rare"]
common_enrichment = results["common"]

In [ ]:
# Total G→X variants in common matrix, per group
r_common = results["common"]
g_row_pos = r_common["counts_pos"].loc["G"].sum()
g_row_neg = r_common["counts_neg"].loc["G"].sum()
print(f"G→anything: pos = {g_row_pos}, neg = {g_row_neg}")
print(f"G→S fraction: pos = {77/g_row_pos:.3f}, neg = {19/g_row_neg:.3f}")

In [ ]:
# Quick check
gs_pos = df_for_rg[
    (df_for_rg["before_aa"] == "G") & 
    (df_for_rg["after_aa"] == "S") & 
    (df_for_rg["group"] == "pos") &
    (df_for_rg["AF_joint"] >= 1e-5) &
    (df_for_rg["Consequence"].str.contains("missense_variant", na=False))
]
gs_neg = df_for_rg[
    (df_for_rg["before_aa"] == "G") & 
    (df_for_rg["after_aa"] == "S") & 
    (df_for_rg["group"] == "neg") &
    (df_for_rg["AF_joint"] >= 1e-5) &
    (df_for_rg["Consequence"].str.contains("missense_variant", na=False))
]
print("G→S codon distribution in pos:")
print(gs_pos["Codons"].value_counts().head(10))
print("\nG→S codon distribution in neg:")
print(gs_neg["Codons"].value_counts().head(10))

In [ ]:
import src.analysis_visualization.codon_usage as cu

results = cu.run_codon_usage_analysis(region_by_id, dataset="gnomad")

# # For your G→S specific question:
# print("\nGlycine codon proportions:")
# print(f"  Pos:  GGC={results['proportions']['pos']['G']['GGC']:.3f}, "
#       f"GGT={results['proportions']['pos']['G']['GGT']:.3f}, "
#       f"GGA={results['proportions']['pos']['G']['GGA']:.3f}, "
#       f"GGG={results['proportions']['pos']['G']['GGG']:.3f}")
# print(f"  Neg:  GGC={results['proportions']['neg']['G']['GGC']:.3f}, "
#       f"GGT={results['proportions']['neg']['G']['GGT']:.3f}, "
#       f"GGA={results['proportions']['neg']['G']['GGA']:.3f}, "
#       f"GGG={results['proportions']['neg']['G']['GGG']:.3f}")
# print(f"  Ref:  GGC={results['reference_proportions']['G']['GGC']:.3f}, "
#       f"GGT={results['reference_proportions']['G']['GGT']:.3f}, "
#       f"GGA={results['reference_proportions']['G']['GGA']:.3f}, "
#       f"GGG={results['reference_proportions']['G']['GGG']:.3f}")

In [ ]:
df_for_rg

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx
# Full pipeline — observed + expected + enrichment + plot
result = smx.run_composition_normalized_analysis(
    df_for_rg,
    region_by_id,
    min_total=5,
    dataset="gnomad",
)

In [ ]:
print("Expected pos matrix (non-zero cells):")
exp_pos = result["exp_pos"]
nonzero_exp = exp_pos[exp_pos > 0]
print(f"  Total cells with expected > 0: {(exp_pos > 0).sum().sum()} / 400")
print(f"  Expected total (should match observed total): {exp_pos.values.sum():.1f}")
print(f"  Observed pos total: {result['obs_pos'].values.sum()}")

print("\nRatios — sample non-NaN pos ratios:")
ratio_pos = result["ratio_pos"]
finite_pos = ratio_pos[ratio_pos.notna()].stack()
print(f"  Cells with finite ratio: {len(finite_pos)} / 400")
print(f"  Ratio summary: median={finite_pos.median():.3f}, "
      f"min={finite_pos.min():.3f}, max={finite_pos.max():.3f}")
print(f"  First 10 values:")
print(finite_pos.head(10))

In [ ]:
 
result = smx.run_codon_substitution_analysis(
    df_for_rg
)

# def run_codon_substitution_analysis(
#     df: pd.DataFrame,
#     group_col: str = "group",
#     pos_label: str = "pos",
#     neg_label: str = "neg",
#     min_total: int = 5,
#     dataset: str = "gnomad",
#     save: bool = True,
#     **plot_kwargs,
# ) -> dict:
#     """
#     [Dataset-agnostic]
#     End-to-end codon substitution matrix: build counts, enrichment, plot.
#     """
#     df_pos = df[df[group_col] == pos_label]
#     df_neg = df[df[group_col] == neg_label]
 
#     counts_pos = compute_codon_substitution_counts(df_pos)
#     counts_neg = compute_codon_substitution_counts(df_neg)
 
#     enrichment = compute_codon_enrichment(counts_pos, counts_neg, min_total=min_total)
#     plot_codon_substitution_matrix(enrichment, dataset=dataset, save=save, **plot_kwargs)
 
#     return enrichment
 

In [ ]:
#### AF analysis

In [ ]:
import src.rg_disruption as rgd

df_rg = rgd.compute_rg_disruption_columns(df_for_rg, region_by_id)

# Sanity check the classification
print("Variants hitting an RG motif:")
print(df_rg["hits_rg"].value_counts())

print("\nOf those, how many actually disrupt it:")
hits = df_rg[df_rg["hits_rg"]]
print(hits["is_rg_disrupting"].value_counts())

print("\nDisruption by type:")
print(hits[hits["is_rg_disrupting"]]["disruption_type"].value_counts())

print("\nR vs G role:")
print(hits["rg_role"].value_counts())

print("\nTop 15 transitions (disrupting only):")
print(hits[hits["is_rg_disrupting"]]["aa_transition"].value_counts().head(15))

print("\nRG hits by group:")
print(df_rg.groupby("group")["hits_rg"].agg(["sum", "count", "mean"]))

print("\nDisruption by group (of variants that hit an RG):")
print(df_rg[df_rg["hits_rg"]].groupby("group")["is_rg_disrupting"].agg(["sum", "count", "mean"]))

In [ ]:
df_rg

In [ ]:
import src.rg_analysis as rga

# 1. The big one: compare disruption rates
fig, stats_dict = rga.plot_rg_disruption_rates(df_rg, dataset="gnomad", missense_only=True)
plt.show()

# 2. Transition heatmap
fig = rga.plot_transition_heatmap(df_rg, dataset="gnomad", missense_only=True)
plt.show()

# 3. gnomAD-only: AF spectrum
fig, af_stats = rga.plot_af_spectrum(df_rg, missense_only=True)
plt.show()